## 4 — Gráficos: Perfil dos Estudantes


| Código | Tipo | Descrição |
|--------|------|-----------|
| 15 | Pirâmide etária | Faixa etária × sexo |
| 16 | Barras horizontais | Distribuição por cor/raça |
| 17 | Barras horizontais | Distribuição por renda familiar |
| 18 | Barras | Distribuição por turno |
| 19 | Heatmap | Evasão — Faixa Etária × Renda Familiar |
| 20 | Heatmap | Evasão — Sexo × Turno |
| 21 | Heatmap | Evasão — Cor/Raça × Renda Familiar |
| 22 | Heatmap | Retenção — Faixa Etária × Renda Familiar |
| 23 | Heatmap | Retenção — Sexo × Turno |
| 24 | Heatmap | Retenção — Cor/Raça × Renda Familiar |
| 25 | Heatmap | Situação (Evasão/Retenção/Conclusão) × Perfil Sociodemográfico |

#### 4.1 Importações, configurações e paletas

In [ ]:

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils import (
    calcular_indicadores,
    gerar_mapa_cores,
    aplicar_layout_light,
    escala_indicador,
    CORES_INDICADORES,
    CORES_CATEGORIA,
    CORES_CONCLUSAO_LIGHT,
    CORES_FLUXO_LIGHT,
    CORES_MATRICULAS_ATIVAS_LIGHT,
    CORES_SITUACAO_LIGHT,
    COR_TEMPO_MEDIANO,
    CORES_CURSO,
    CORES_SITUACAO,
    PALETA_QUALITATIVA_LIGHT,
    ROTULOS_INDICADORES,
)

ORDEM_ETARIA = [
    "Menor de 14 anos", "15 a 19 anos", "20 a 24 anos", "25 a 29 anos", "30 a 34 anos",
    "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos", "Maior de 60 anos",
]
ORDEM_RENDA = ["0<RFP<=0,5", "0,5<RFP<=1", "1<RFP<=1,5", "1,5<RFP<=2,5", "2,5<RFP<=3,5", "RFP>3,5", "Não declarada"]
CORES_SEXO = {"F": "#AD1457", "M": "#00838F",}

def pivot_indicador_perfil(
    df_base,
    col_y,
    col_x,
    indicador,
    ordem_y=None,
    ordem_x=None,
    n_minimo=1,
):
    ind = calcular_indicadores(df_base, [col_y, col_x])

    ind.loc[ind["matr_atendidas"] < n_minimo, indicador] = pd.NA

    pivot = ind.pivot_table(
        index=col_y,
        columns=col_x,
        values=indicador,
        aggfunc="mean",
    )

    if ordem_y:
        pivot = pivot.reindex([v for v in ordem_y if v in pivot.index])

    if ordem_x:
        pivot = pivot[[v for v in ordem_x if v in pivot.columns]]

    return pivot


def fig_heatmap_perfil(pivot, indicador):
    fig = px.imshow(
        pivot,
        text_auto=".1f",
        color_continuous_scale=escala_indicador(indicador),
        labels={"color": f"{ROTULOS_INDICADORES[indicador]} (%)"},
        aspect="auto",
    )
    aplicar_layout_light(fig, altura=430)
    return fig

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

#### 4.2 Carga dos dados tratados

In [11]:
df_completo = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")
print("Shape:", df_completo.shape)
df_completo.head(3)

Shape: (10285, 18)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino


#### 4.3 Filtros utilizados

In [12]:
ano_selecionado = int(df_completo['Ano'].max())
tipos_selecionados = sorted(df_completo['Tipo de Curso'].dropna().unique())
cursos_selecionados = sorted(df_completo['Nome de Curso'].dropna().unique())
n_minimo = 5

df = df_completo[
    (df_completo['Ano'] == ano_selecionado)
    & (df_completo['Tipo de Curso'].isin(tipos_selecionados))
    & (df_completo['Nome de Curso'].isin(cursos_selecionados))
].copy()

print('Ano selecionado:', ano_selecionado)
print('Matrículas distintas:', df['Código da Matricula'].nunique())
print('Cursos:', df['Nome de Curso'].nunique())

Ano selecionado: 2024
Matrículas distintas: 1745
Cursos: 11


#### 4.4 Gráficos

#### Gráfico 15 — Pirâmide etária por sexo

In [15]:
faixas_presentes = [f for f in ORDEM_ETARIA if f in df['Faixa Etária'].dropna().unique()]
piramide_df = df.groupby(['Faixa Etária', 'Sexo'])['Código da Matricula'].nunique().reset_index(name='Qtd')
piramide_df['Faixa Etária'] = pd.Categorical(piramide_df['Faixa Etária'], categories=faixas_presentes, ordered=True)
piramide_df = piramide_df.sort_values('Faixa Etária')
fig_g15 = go.Figure()
sexos = sorted(piramide_df['Sexo'].dropna().unique())
for sexo in sexos:
    subset = piramide_df[piramide_df['Sexo'] == sexo]
    multiplicador = -1 if sexo == sexos[0] else 1
    fig_g15.add_trace(go.Bar(name=sexo, y=subset['Faixa Etária'].astype(str), x=subset['Qtd'] * multiplicador, orientation='h', marker_color=CORES_SEXO.get(sexo, '#757575'), text=subset['Qtd'].astype(str), textposition='inside'))
valor_max = int(piramide_df['Qtd'].max()) if len(piramide_df) else 10
ticks = list(range(-valor_max, valor_max + max(1, valor_max // 5), max(1, valor_max // 5)))
fig_g15.update_layout(barmode='relative', xaxis=dict(tickvals=ticks, ticktext=[str(abs(v)) for v in ticks], title='Matrículas'), yaxis=dict(title=''))
aplicar_layout_light(fig_g15, altura=430)
fig_g15.show()

piramide_df.head()

,Faixa Etária,Sexo,Qtd
0,15 a 19 anos,F,141
1,15 a 19 anos,M,153
2,20 a 24 anos,F,151
3,20 a 24 anos,M,216
4,25 a 29 anos,F,155


#### Gráficos 16 a 18 — Distribuições gerais

In [16]:
raca_df = df["Cor / Raça"].value_counts().reset_index()
raca_df.columns = ["Cor/Raça", "Qtd"]
raca_df["Percentual"] = raca_df["Qtd"] / raca_df["Qtd"].sum() * 100
raca_df["Texto"] = raca_df.apply(
    lambda linha: f"{int(linha['Qtd'])} ({linha['Percentual']:.1f}%)",
    axis=1,
)

fig_g16 = px.bar(
    raca_df.sort_values("Qtd"),
    x="Qtd",
    y="Cor/Raça",
    orientation="h",
    color="Qtd",
    color_continuous_scale="YlGnBu",
    text="Texto",
    labels={"Qtd": "Matrículas"},
)
fig_g16.update_traces(textposition="outside", cliponaxis=False)
fig_g16.update_xaxes(range=[0, raca_df["Qtd"].max() * 1.25])
fig_g16.update_layout(coloraxis_showscale=False)
aplicar_layout_light(fig_g16, altura=430)
fig_g16.show()

raca_df.head()


,Cor/Raça,Qtd,Percentual,Texto
0,Branca,990,56.73,990 (56.7%)
1,Preta,358,20.52,358 (20.5%)
2,Parda,293,16.79,293 (16.8%)
3,Não declarada,94,5.39,94 (5.4%)
4,Indígena,8,0.46,8 (0.5%)


In [17]:
renda_df = (
    df["Renda Familiar"]
    .value_counts()
    .reindex([r for r in ORDEM_RENDA if r in df["Renda Familiar"].unique()])
    .reset_index()
)
renda_df.columns = ["Renda", "Qtd"]
renda_df["Percentual"] = renda_df["Qtd"] / renda_df["Qtd"].sum() * 100
renda_df["Texto"] = renda_df.apply(
    lambda linha: f"{int(linha['Qtd'])} ({linha['Percentual']:.1f}%)",
    axis=1,
)

fig_g17 = px.bar(
    renda_df.sort_values("Qtd"),
    x="Qtd",
    y="Renda",
    orientation="h",
    color="Qtd",
    color_continuous_scale="YlGnBu",
    text="Texto",
    labels={"Qtd": "Matrículas"},
)
fig_g17.update_traces(textposition="outside", cliponaxis=False)
fig_g17.update_xaxes(range=[0, renda_df["Qtd"].max() * 1.25])
fig_g17.update_layout(coloraxis_showscale=False)
aplicar_layout_light(fig_g17, altura=430)
fig_g17.show()

renda_df.head()

,Renda,Qtd,Percentual,Texto
0,"0<RFP<=0,5",310,17.77,310 (17.8%)
1,"0,5<RFP<=1",383,21.95,383 (21.9%)
2,"1<RFP<=1,5",417,23.90,417 (23.9%)
3,"1,5<RFP<=2,5",216,12.38,216 (12.4%)
4,"2,5<RFP<=3,5",86,4.93,86 (4.9%)


In [18]:
turno_df = df["Turno"].value_counts().reset_index()
turno_df.columns = ["Turno", "Qtd"]
turno_df["Percentual"] = turno_df["Qtd"] / turno_df["Qtd"].sum() * 100
turno_df["Texto"] = turno_df.apply(
    lambda linha: f"{int(linha['Qtd'])} ({linha['Percentual']:.1f}%)",
    axis=1,
)

fig_g18 = px.bar(
    turno_df,
    x="Turno",
    y="Qtd",
    color="Turno",
    color_discrete_map=gerar_mapa_cores(turno_df["Turno"]),
    text="Texto",
    labels={"Qtd": "Matrículas"},
)
fig_g18.update_traces(textposition="outside", cliponaxis=False)
fig_g18.update_layout(coloraxis_showscale=False)
aplicar_layout_light(fig_g18, altura=430)
fig_g18.show()

turno_df.head()

,Turno,Qtd,Percentual,Texto
0,Noturno,1026,58.80,1026 (58.8%)
1,Matutino,449,25.73,449 (25.7%)
2,Vespertino,179,10.26,179 (10.3%)
3,Integral,91,5.21,91 (5.2%)


#### Gráficos 19 a 21 — Heatmaps de evasão

Células com menos de `n_minimo` matrículas são deixadas em branco.

In [37]:
p19 = pivot_indicador_perfil(df, "Faixa Etária", "Renda Familiar", "TE", ORDEM_ETARIA, ORDEM_RENDA, n_minimo)
fig_g19 = fig_heatmap_perfil(p19, "TE")
fig_g19.show()

p20 = pivot_indicador_perfil(df, "Sexo", "Turno", "TE", n_minimo=n_minimo)
fig_g20 = fig_heatmap_perfil(p20, "TE")
fig_g20.show()

p21 = pivot_indicador_perfil(df, "Cor / Raça", "Renda Familiar", "TE", ordem_x=ORDEM_RENDA, n_minimo=n_minimo)
fig_g21 = fig_heatmap_perfil(p21, "TE")
fig_g21.show()

#### Gráficos 22 a 24 — Heatmaps de retenção

In [42]:
p22 = pivot_indicador_perfil(df, "Faixa Etária", "Renda Familiar", "TR", ORDEM_ETARIA, ORDEM_RENDA, n_minimo)
fig_g22 = fig_heatmap_perfil(p22, "TR")
fig_g22.show()

p23 = pivot_indicador_perfil(df, "Sexo", "Turno", "TR", n_minimo=n_minimo)
fig_g23 = fig_heatmap_perfil(p23, "TR")
fig_g23.show()

p24 = pivot_indicador_perfil(df, "Cor / Raça", "Renda Familiar", "TR", ordem_x=ORDEM_RENDA, n_minimo=n_minimo)
fig_g24 = fig_heatmap_perfil(p24, "TR")
fig_g24.show()

#### Gráfico 25 — Matriz consolidada Situação × Perfil

Esta matriz mostra, para cada categoria de perfil, a composição percentual das situações de matrícula.

In [45]:
variaveis = ["Sexo", "Turno", "Renda Familiar", "Faixa Etária", "Cor / Raça"]
indicadores_situacao = {"Evasão": "TE", "Retenção": "TR", "Conclusão": "TC"}

linhas = []
for variavel in variaveis:
    ind_variavel = calcular_indicadores(df, [variavel])
    ind_variavel = ind_variavel[ind_variavel["matr_atendidas"] >= n_minimo]

    for _, linha_ind in ind_variavel.iterrows():
        categoria = linha_ind[variavel]

        if pd.isna(categoria):
            continue

        linha = {"Perfil": f"{variavel}: {categoria}"}
        for rotulo, indicador_coluna in indicadores_situacao.items():
            linha[rotulo] = round(linha_ind[indicador_coluna], 1)

        linhas.append(linha)

consolidado = pd.DataFrame(linhas)

if not consolidado.empty:
    matriz = consolidado.set_index("Perfil")[list(indicadores_situacao.keys())].T

    fig_25 = px.imshow(
        matriz,
        text_auto=".1f",
        color_continuous_scale="Purples",
        labels={"color": "Taxa (%)"},
        aspect="auto",
    )

    colunas = list(matriz.columns)
    separadores = []
    variavel_atual = colunas[0].split(":")[0]

    for i, col in enumerate(colunas[1:], start=1):
        variavel_nova = col.split(":")[0]
        if variavel_nova != variavel_atual:
            separadores.append(i)
            variavel_atual = variavel_nova

    shapes = [
        dict(
            type="line",
            xref="x",
            yref="paper",
            x0=sep - 0.5,
            x1=sep - 0.5,
            y0=0,
            y1=1,
            line=dict(color="#263238", width=3),
        )
        for sep in separadores
    ]

    aplicar_layout_light(fig_25, altura=420)

    fig_25.update_layout(
        xaxis_title="Categoria sociodemográfica",
        yaxis_title="Situação",
        shapes=shapes,
    )
    fig_25.update_xaxes(tickangle=-35)
    fig_25.show()
else:
    print("Não há categorias com tamanho mínimo suficiente para exibir a visão consolidada.")
    
matriz.head()

Perfil,Sexo: F,Sexo: M,Turno: Integral,Turno: Matutino,Turno: Noturno,Turno: Vespertino,"Renda Familiar: 0,5<RFP<=1","Renda Familiar: 0<RFP<=0,5","Renda Familiar: 1,5<RFP<=2,5","Renda Familiar: 1<RFP<=1,5","Renda Familiar: 2,5<RFP<=3,5",Renda Familiar: Não declarada,"Renda Familiar: RFP>3,5",Faixa Etária: 15 a 19 anos,Faixa Etária: 20 a 24 anos,Faixa Etária: 25 a 29 anos,Faixa Etária: 30 a 34 anos,Faixa Etária: 35 a 39 anos,Faixa Etária: 40 a 44 anos,Faixa Etária: 45 a 49 anos,Faixa Etária: 50 a 54 anos,Faixa Etária: 55 a 59 anos,Faixa Etária: Maior de 60 anos,Cor / Raça: Branca,Cor / Raça: Indígena,Cor / Raça: Não declarada,Cor / Raça: Parda,Cor / Raça: Preta
Evasão,2.20,2.40,3.30,2.70,2.10,2.20,1.80,2.90,2.80,1.40,3.50,1.90,7.60,4.10,3.00,1.70,2.20,2.70,2.40,0.00,1.50,0.00,0.00,2.40,0.00,2.10,3.10,1.70
Retenção,32.10,38.10,15.40,38.50,36.80,29.10,38.10,42.30,31.90,32.10,29.10,33.30,34.90,0.30,32.40,53.60,42.00,41.10,38.90,37.60,58.80,30.40,52.40,34.50,37.50,34.00,35.10,38.30
Conclusão,5.30,5.70,4.40,4.70,4.30,15.60,4.20,5.50,9.30,5.30,8.10,4.90,3.00,7.10,5.20,4.20,5.50,7.50,4.80,4.30,2.90,8.70,9.50,5.70,12.50,8.50,6.10,3.60
